In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = os.getenv("CLAUDE_MODEL")

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json

# For step 2, where we create an evaluation dataset. We ask Claude to do it for us.
def generate_dataset():
    prompt = """
		Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
		that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
		each representing task that requires Python, JSON, or a Regex to complete.

		Example output:
		```json
		[
			{
				"task": "Description of task",
			},
			...additional
		]
		```

		* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
		* Focus on tasks that do not require writing much code

		Please generate 3 objects.
	"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()

with open("dataset.json", "w") as file:
  json.dump(dataset, file, indent=2)

dataset

[{'task': "Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/object') and returns a dictionary with 'bucket' and 'key' as separate fields."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific DynamoDB table named 'users-table'."},
 {'task': "Write a regular expression that matches valid AWS EC2 instance IDs (format: i-followed by 17 hexadecimal characters, e.g., 'i-0a1b2c3d4e5f6g7h8')."}]

In [4]:
def run_prompt(test_case):
	"""Merges the prompt and test case input, then returns the result"""

	prompt = f"""
		Please solve the following task:
		{test_case["task"]}
	"""

	messages = []
	add_user_message(messages, prompt)
	output = chat(messages)
	return output

In [5]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
		You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

		Original Task:
		<task>
		{test_case["task"]}
		</task>

		Solution to Evaluate:
		<solution>
		{output}
		</solution>

		Output Format
		Provide your evaluation as a structured JSON object with the following fields, in this specific order:
		- "strengths": An array of 1-3 key strengths
		- "weaknesses": An array of 1-3 key areas for improvement
		- "reasoning": A concise explanation of your overall assessment
		- "score": A number between 1-10

		Respond with JSON. Keep your response concise and direct.
		Example response shape:
		{{
			"strengths": string[],
			"weaknesses": string[],
			"reasoning": string,
			"score": number
		}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)
  

In [6]:
def run_test_case(test_case):
	"""Calls run_prompt, then grades the result"""
	output = run_prompt(test_case)

	# TODO - Grading
	model_grade = grade_by_model(test_case, output)
	score = model_grade["score"]
	reasoning = model_grade["reasoning"]

	return {
		"output" : output,
		"test_case" : test_case,
		"score" : score,
		"reasoning" : reasoning
	}


In [7]:
from statistics import mean

def run_eval(dataset):
	"""Loads the dataset and calls run_test_case with each case"""
	results = []

	for test_case in dataset:
		result = run_test_case(test_case)
		results.append(result)

	average_score = mean([result["score"] for result in results])
	print(f"Average score: {average_score}")
	return results

In [9]:
with open("dataset.json", "r") as file:
	dataset = json.load(file)

results = run_eval(dataset)

In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a comprehensive solution with multiple approaches:\n\n## Basic Solution\n\n```python\ndef parse_s3_uri(uri: str) -> dict:\n    \"\"\"\n    Parse an AWS S3 bucket URI and return bucket and key.\n    \n    Args:\n        uri: S3 URI in format 's3://bucket-name/path/to/object'\n        \n    Returns:\n        Dictionary with 'bucket' and 'key' fields\n        \n    Raises:\n        ValueError: If URI format is invalid\n    \"\"\"\n    if not uri.startswith('s3://'):\n        raise ValueError(\"URI must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = uri[5:]\n    \n    # Split by first '/'\n    parts = uri_without_prefix.split('/', 1)\n    \n    if len(parts) < 1 or not parts[0]:\n        raise ValueError(\"Invalid S3 URI: missing bucket name\")\n    \n    bucket = parts[0]\n    key = parts[1] if len(parts) > 1 else ''\n    \n    return {\n        'bucket': bucket,\n        'key': key\n    }\n\n\n# Test c